In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import time
from collections import deque
import numpy as np
import h5py
import plotly.graph_objects as go
import gc

from IPython.display import display, clear_output
import ipywidgets as widgets

# load the data
n_samples_to_load = 1000
with h5py.File("scene_collections/phase_1_20k.h5", "r") as f:
    data = [{key: np.array(f[item][key]) for key in f[item]} for item in list(f.keys())[:n_samples_to_load]]

torch.cuda.is_available()

False

In [ ]:
# plotting convenience -------------------------------------------------------------------------------------------------
# Function to safely clear existing plotly widgets
def reset_fig(plotflag):
    if plotflag is False:
        return
    
    if "fig" in globals():
        del globals()["fig"]
    gc.collect()

# helper function to initialize the plots 
def initialize_plots(plotflag):
    if plotflag is False:
        return 
    
    fig = go.FigureWidget()
    # add dummy trace so that the legend is displayed from the beginning on
    trace_training   = fig.add_scatter(x=[None], y=[None], mode="lines", name="train. loss", 
                                       line=dict(color="firebrick", width=2)
                                       )
    # add dummy trace so that the legend is displayed from the beginning on
    trace_validation = fig.add_scatter(x=[None], y=[None], mode="lines", name="valid. loss", 
                                       line=dict(color="SeaGreen", width=2)
                                       )
    # horizontal line for tracking the current best validation loss   
    trace_best_loss  = fig.add_scatter(x=[None], y=[None], mode="lines+markers", name="Best val. loss", 
                                       line=dict(color="SeaGreen", width=2, dash="dot"), 
                                       marker=dict(size=[0, 18], opacity=[0, 1], symbol=["circle", "diamond-wide"]), 
                                       showlegend=False
                                       )

    # layout settings
    fig.update_layout(height=600, plot_bgcolor="white", margin_pad=10,
                      legend=dict(font=dict(family="Lucida Console, monospace", size=14, color="grey")))
    
    fig.update_xaxes(title="number of samples seen (running total over all epochs)",
                     title_standoff=30,
                     title_font_color="grey", title_font_family="Lucida Console, monospace", title_font_size=20, 
                     zeroline=True, zerolinewidth=3, zerolinecolor="gray",
                     showgrid=True, gridwidth=1.5, gridcolor="lightgray", griddash="dash",
                     tickfont=dict(family="Lucida Console, monospace", size=12, color="grey"),
                     range=[-0.03 * samples_tota, 1.03 * samples_tota])
    
    fig.update_yaxes(title="loss (distance to leak)",
                     title_standoff=30,  
                     title_font_color="grey", title_font_family="Lucida Console, monospace", title_font_size=20,
                     zeroline=True, zerolinewidth=3, zerolinecolor="gray",
                     showgrid=True, gridwidth = 1.5, gridcolor="lightgray",
                     tickfont=dict(family="Lucida Console, monospace", size=12, color="grey"),
                     range=[-0.1, 1.1])
    
    display(fig)
       
    return fig
       
# update training error plot data
def update_training_error_plot(plotflag):
    if plotflag is False:
        return
    
    with fig.batch_update():
        fig.data[0].x += (samples_seen,)  # Update x-axis for training trace
        fig.data[0].y += (loss_batch.item(),)  # Update y-axis for training trace  
        fig.update_layout(
            annotations=[
                # annotation for the processing speed
                dict(
                    x=0.5, y=1.1,
                    xref="paper", yref="paper",
                    text=f"Samples/sec (moving avg): <b>{samples_per_second:.2f}</b>",
                    showarrow=False,
                    font=dict(family="Lucida Console, monospace", size=16, color="gray"),
                    xanchor="center",
                ),
                # annotation for the current best validation loss
                dict(
                    x=1.03 * samples_tota,
                    y=val_loss_best,
                    xref="x", yref="y",
                    text=f"<b> val. best <br> {val_loss_best:09.4f}</b>",
                    showarrow=False,
                    font=dict(family="Lucida Console, monospace", size=14, color="SeaGreen"),
                    xanchor="left",
                    yanchor="middle",
                    align="left"
                ),  
            ]
        )
    # Display updated figure
    clear_output(wait=True)
    display(fig)
    
    return
  
# update validation error plot data
def update_validation_error_plot(plotflag):
    if plotflag is False:
        return 
    
    with fig.batch_update():
        # updates the validation loss trace (all losses over processed samples)
        fig.data[1].x += (samples_seen, )  # Update x-axis for validation trace
        fig.data[1].y += (val_loss, )  # Update y-axis for validation trace   
         
        # Update the horizontal line indicator of best validation loss
        fig.data[2].x = [0, 1.03 * samples_tota]  # Extend the line across the x-axis
        fig.data[2].y = [val_loss_best, val_loss_best]  # Same y value for a horizontal line
        
        # Update the annotation showing the current best validation loss
        fig.update_layout(
            annotations=[
                # annotation for the processing speed
                dict(
                    x=0.5, y=1.1,
                    xref="paper", yref="paper",
                    text=f"Samples/sec (moving avg): <b>{samples_per_second:.2f}</b>",
                    showarrow=False,
                    font=dict(family="Lucida Console, monospace", size=16, color="gray"),
                    xanchor="center",
                ),
                # annotation for the best validation loss
                dict(
                    x=1.03 * samples_tota,
                    y=val_loss_best,
                    xref="x", yref="y",
                    text=f"<b> val. best <br> {val_loss_best:09.4f}</b>",
                    showarrow=False,
                    font=dict(family="Lucida Console, monospace", size=14, color="SeaGreen"),
                    xanchor="left",
                    yanchor="middle",
                    align="left"
                ),
            ]  
        )      
          
    # Display updated figure
    clear_output(wait=True)
    display(fig)
    
    return  
 
# calculate batch processing time  
def calculate_samples_per_second_processed():
    batch_time  = batch_end_time - batch_start_time
    batch_speed = actual_bs / batch_time
    processing_speed.append(batch_speed)
    
    if processing_speed:
        avg_samples_per_second = sum(processing_speed) / len(processing_speed)
        
    return avg_samples_per_second
  
# function for calculating validation error  
def do_validation():
    model.eval()
    with torch.no_grad():  # Turn off gradient calculations for validation
        val_loss = 0
        counter  = 0
        for sample in loader_valid:
            # move batch to device
            sample = {key: value.to(device) for key, value in sample.items()}
            pred = model(sample)
            val_loss_batch = torch.norm(sample["pos_leaks"].squeeze() - pred, p=2, dim=1)
            val_loss      += torch.mean(val_loss_batch)
            counter       += 1
        val_loss = val_loss / counter
        
    return val_loss.item()



# model classes --------------------------------------------------------------------------------------------------------
class MyDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        return sample 
 
# just flatten the whole input and pass it to an mlp (just learns noise)
class FirstDumbMLP(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        
        self.lin1 = nn.Linear(3200, 500)
        self.lin2 = nn.Linear( 500,  50)
        self.lin3 = nn.Linear(  50,   2)
        
        self.activation = nn.ReLU()
        
        self.dropout = nn.Dropout(p=0)
        
        
    def forward(self, sample):
        # static data
        n_sensors_max = 10
        pos_sensors   = sample["pos_sensors"]
        
        # time series data
        wind_direc  = sample["wind_direc"]
        wind_speed  = sample["wind_speed"]
        con_sensors = sample["con_sensors"]
        
        # add some noise to the wind patterns
        noise_stren = 0.1
        noise_direc = torch.rand(wind_direc.shape).to(device)
        noise_speed = torch.rand(wind_speed.shape).to(device)
        wind_direc += noise_stren * noise_direc 
        wind_speed += noise_stren * noise_speed
        
        # reformat input vector
        for t in sample["time"][0]:
            if t == 0:
                input = torch.cat((
                    wind_direc[:, t].unsqueeze(1),
                    wind_speed[:, t].unsqueeze(1),
                    pos_sensors.view(-1, n_sensors_max * 2),
                    con_sensors[:, t]
                    ), 1)
            else:
                temp = torch.cat((
                    wind_direc[:, t].unsqueeze(1),
                    wind_speed[:, t].unsqueeze(1),
                    pos_sensors.view(-1, n_sensors_max * 2),
                    con_sensors[:, t]
                    ), 1)
                input = torch.cat((input, temp), 1)
        
        x = input.float()
                
        # do forward pass through network
        x = self.dropout(self.activation(self.lin1(x)))
        x = self.dropout(self.activation(self.lin2(x)))
        x = self.lin3(x)
    
        #output is literally just x, y position of the leak
        return x
 
# basic attention block with optional masking
class SelfAttention(nn.Module):
    def __init__(self, input_feats_dim, latent_dim, output_feats_dim) -> None:
        """
        input_feats are the main data. latent_dim can be chosen.
        -> note, needs skip connection on the outside!
        -> note, this is modified from standard. Normally input_feats_dim = output_feats_dim!
        """
        super().__init__()
        
        # define dimensions
        self.input_feats_dim  = input_feats_dim
        self.latent_dim       = latent_dim
        self.output_feats_dim = output_feats_dim
        
        # define attention weights (= matrices Wq, Wk, Wv)
        self.query = nn.Linear(self.input_feats_dim, self.latent_dim)
        self.key   = nn.Linear(self.input_feats_dim, self.latent_dim)
        self.value = nn.Linear(self.input_feats_dim, self.latent_dim)
        
        # define last linear reshaping layer (= matrix Wout)
        self.linear_out = nn.Linear(self.latent_dim, self.output_feats_dim)
        
        # softmax (row-wise!)
        self.softmax = nn.Softmax(dim=2) 
        
        
    def forward(self, x, msk=None):
        """
        input x  : [BS (var, broadcastable) | input length (var) | feats (known, Architecture defining)]
        input msk: [BS (var, broadcastable) | input length (var) ]
        """

        # fallback for when no mask is provided -> [BS (var, broadcastable) | 1 | feats (known, architecture def.)]
        # (so the maskvector is a lying row vector times batch size)
        if msk is None:
            msk = torch.zeros(x.shape[0], x.shape[1]).to(device)
        
        # calculate the embedded vectors in latent space (= matrices Q, K, V)
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # calculate the attention matrix -> normalize by latent dimension -> apply softmax -> and weigh it with V
        # mask has [BS, maskvec]. needs unsqueeze(1) so that it is a [BS, 1, maskvec] "lying" row vector -> broadcasted
        SCO = torch.bmm(Q, K.transpose(1, 2)) / (self.latent_dim**0.5)
        SCO = SCO.masked_fill(msk.unsqueeze(1) == 1, float("-inf"))
        ATT = self.softmax(SCO)
        WGH = torch.bmm(ATT, V)
        
        # apply last linear layer to reshape output
        x = self.linear_out(WGH)
        
        return x 
  
# basic attention pooling block with optional masking
class AttentionPooling(nn.Module):
    def __init__(self, input_feats_dim, hidden_layers=[128]) -> None:
        """
        Implements the attention pooling mechanism. A simple mlp learns one weight per input sequence. The output is then a linear combination of all the weights and inputs.
        """
        super().__init__()
        
        self.activation = nn.ReLU()
        self.softmax    = nn.Softmax(dim=1)
        
        # set up the sequential mlp to learn the per-input weights [BS | input length | 1]
        layers        = []
        current_feats = input_feats_dim
        for el in hidden_layers:
            layers.append(nn.Linear(current_feats, el))
            layers.append(self.activation)
            current_feats = el
        layers.append(nn.Linear(current_feats, 1))
        self.mlp = nn.Sequential(*layers)
        
    def forward(self, x, msk=None):
        """
        input x:   [BS (var, broadcastable) | input length (var) | n feats (known, architecture defining)]  
        input msk: [BS (var, broadcastable) | input length (var) ]
        """
        
        # fallback for when no mask is provided -> [BS (var, broadcastable) | 1 | feats (known, architecture def.)]
        # (so the maskvector is a lying row vector times batch size)
        if msk is None:
            msk = torch.zeros(x.shape[0], x.shape[1]).to(device)
        
        # calculate the weights by learning them with an mlp    
        d = self.mlp(x)
        d = d.masked_fill(msk.unsqueeze(2) == 1, float("-inf"))
        d = self.softmax(d) # d1 = torch.ones(d.shape).to(device) * 0.01
        
        # recombine the outputs from a linear combination of weights and inputs
        x = torch.bmm(d.permute(0, 2, 1), x)
        
        return x
    
# working attention network!
class WithDoubleAttention(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        
        self.linear_1x1           = nn.Linear( 5, 20)
        self.linear_1x2           = nn.Linear(20, 20)
        self.scene_attention_1    = SelfAttention(20, 20, 20)
        self.linear_1x3           = nn.Linear(20, 20)
        self.scene_attention_2    = SelfAttention(20, 20, 20)
        self.scene_attention_pool = AttentionPooling(20, [20])
        
        self.linear_2x1           = nn.Linear( 20, 100)
        self.linear_2x2           = nn.Linear(100, 100)
        self.time_attention_1     = SelfAttention(100, 100, 100)
        self.linear_2x3           = nn.Linear(100, 100)
        self.time_attention_2     = SelfAttention(100, 100, 100)
        self.time_attention_pool  = AttentionPooling(100, [100])
                
        self.linear_3x1           = nn.Linear(100, 100)
        self.linear_3x2           = nn.Linear(100,  50)
        self.linear_3x3           = nn.Linear( 50,   2)
        
        self.activation = nn.ReLU()
        
    def forward(self, sample):   
        # data preparation ---------------------------------------------------------------------------------------------
        # static data
        n_sensors_max = 10
        pos_sensors   = sample["pos_sensors"]
        msk_sensors   = sample["msk_sensors"].unsqueeze(1).repeat(1, 100, 1) # to match the shape for attention layer
        # time series data
        wind_direc    = sample["wind_direc"]
        wind_speed    = sample["wind_speed"]
        con_sensors   = sample["con_sensors"]
        
        # reformat input vector
        input = None
        for t in sample["time"][0]:
            if input is None:
                input = torch.cat((
                    pos_sensors.unsqueeze(1),
                    con_sensors[:, t, :].unsqueeze(1).unsqueeze(3),
                    wind_direc[:, t].unsqueeze(1).unsqueeze(2).unsqueeze(3).expand((-1, 1, n_sensors_max, 1)),
                    wind_speed[:, t].unsqueeze(1).unsqueeze(2).unsqueeze(3).expand((-1, 1, n_sensors_max, 1)),
                ), dim=3)
            else:
                temp = torch.cat((
                    pos_sensors.unsqueeze(1),
                    con_sensors[:, t, :].unsqueeze(1).unsqueeze(3),
                    wind_direc[:, t].unsqueeze(1).unsqueeze(2).unsqueeze(3).expand((-1, 1, n_sensors_max, 1)),
                    wind_speed[:, t].unsqueeze(1).unsqueeze(2).unsqueeze(3).expand((-1, 1, n_sensors_max, 1)),
                ), dim=3)
                input = torch.cat((input, temp), dim=1)
                
        # in the form of [BS, Time, Sensor_nr, features_per_sensor]
        input1 = input.float()
        # # in the form of [BS, Time, (Sensor_nr*features_per_sensor)]
        # input2 = input1.view(-1, 100, 50)
        
        # select correctly shaped input
        x = input1
        M = msk_sensors
        
        # aggregate all the broadcasting dimensions in one [BS*time, n_sensors, n_feats]
        x = x.reshape(-1, 10, 5)
        M = M.reshape(-1, 10)

        # forward function ---------------------------------------------------------------------------------------------
        
        # scene-elementwise linear layers 
        x = self.activation(self.linear_1x1(x))
        x = self.activation(self.linear_1x2(x))       
        # apply scene-wide attention Layer
        x = self.scene_attention_1(x) + x
        # scene-elementwise linear layers
        x = self.activation(self.linear_1x3(x))
        # apply scene-wide attention Layer
        x = self.scene_attention_2(x) + x
        # apply scene-wide attention pooling (squeeze really important because output is [BS, 1, n_feats=20])
        x = self.scene_attention_pool(x, M).squeeze()
        
        # redistribute time dimension again (now only need to broadcast along BS dimension)
        x = x.reshape(-1, 100, 20)
        
        # time-elementwise linear layers
        x = self.activation(self.linear_2x1(x))
        x = self.activation(self.linear_2x2(x))
        # apply time-wide attention layer
        x = self.time_attention_1(x) + x
        # time-elementwise linear layers
        x = self.activation(self.linear_2x3(x))
        # apply time-wide attention layer
        x = self.time_attention_2(x) + x
        # apply time-wide attention pooling (squeeze really important because output is [BS, 1, n_feats=100])
        x = self.time_attention_pool(x).squeeze()   
        
        # MLP at the end
        x = self.activation(self.linear_3x1(x))
        x = self.activation(self.linear_3x2(x))
        x = self.linear_3x3(x)
        
        return x



# set up training and data ---------------------------------------------------------------------------------------------
device  = "cuda:0"

# dataloaders
batch_size   = 256
dataset      = MyDataset(data)
L            = len(dataset)
dataset_train, dataset_valid, dataset_testt = random_split(dataset, [int(0.8*L), int(0.1*L), L-int(0.8*L)-int(0.1*L)])
loader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
loader_valid = DataLoader(dataset_valid, batch_size=batch_size, shuffle=True)
loader_testt = DataLoader(dataset_testt, batch_size=batch_size, shuffle=True)

# initialize model and model weights and move to device
n_epochs = 50
model    = WithDoubleAttention().to(device)
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)
optim    = torch.optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999), weight_decay=0.0)

# initialize tracking variables
samples_seen       = 0
samples_tota       = n_epochs * len(dataset_train)
processing_speed   = deque(maxlen=10)
samples_per_second = 0



# Initialize the plot --------------------------------------------------------------------------------------------------
plotflag = True
reset_fig(plotflag)
fig = initialize_plots(plotflag)



# Training loop --------------------------------------------------------------------------------------------------------
# do an initial validation
val_loss      = do_validation()
val_loss_best = val_loss
# update plot
update_validation_error_plot(plotflag)

for E in range(n_epochs):   
    
    model.train()
    
    for sample in loader_train:
        # get the actual batch size
        actual_bs     = len(sample["n_sensors"]) # use some quantity, where the batch size can be easily seen       
        samples_seen += actual_bs
        # move batch to device
        sample = {key: value.to(device) for key, value in sample.items()}
        # start timer for batch
        batch_start_time = time.time()
        
        # Zero gradients
        optim.zero_grad()
        # Forward pass
        pred = model(sample)
        # Compute loss
        loss       = torch.norm(sample["pos_leaks"].squeeze() - pred, p=2, dim=1)
        loss_batch = torch.mean(loss)
        # Compute gradients
        loss_batch.backward()
        # Update weights
        optim.step()
        
        # end timer for batch
        batch_end_time = time.time()
        
        # update the plot
        samples_per_second = calculate_samples_per_second_processed()
        update_training_error_plot(plotflag)

    
    # validation at end of each epoch    
    val_loss = do_validation()
    if val_loss < val_loss_best:
        val_loss_best = val_loss
    # update plot
    update_validation_error_plot(plotflag)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import time
from collections import deque
import numpy as np
import h5py
import plotly.graph_objects as go
import gc

from IPython.display import display, clear_output
import ipywidgets as widgets

# load the data
n_samples_to_load = 200
with h5py.File("scene_collections/2024-11-15--21-59-16--X20000.h5", "r") as f:
    data = [{key: np.array(f[item][key]) for key in f[item]} for item in list(f.keys())[:n_samples_to_load]]

torch.cuda.is_available()



# model classes --------------------------------------------------------------------------------------------------------
class MyDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        return sample 
 


# datasets (untouched...)
batch_size   = 256
dataset      = MyDataset(data)
L            = len(dataset)
dataset_train, dataset_valid, dataset_testt = random_split(dataset, [int(0.8*L), int(0.1*L), L-int(0.8*L)-int(0.1*L)])


loader_train = DataLoader(dataset_train, batch_size=batch_size, shuffle=True)
loader_valid = DataLoader(dataset_valid, batch_size=batch_size, shuffle=True)
loader_testt = DataLoader(dataset_testt, batch_size=batch_size, shuffle=True)





In [27]:
from torch.utils.data import DataLoader, Dataset, Sampler
import math
import random


class ScheduledBatchSampler(Sampler):
    """
    batch_sampler with batch size scheduling based on a custom dict. Use as custom batch_sampler fn in Dataloader().
    batch schedule = {how many epochs : batch size, ... : ...}
    """
    
    def __init__(self, data_source, schedule, shuffle=False, drop_last=False):
        """
        Args:
            data_source (Dataset): Dataset to sample from.
            schedule (dict)      : mapping of nr of epochs to batch size e.g. {10 : 128, 20 : 256, 50 : 512}
            shuffle (bool)       : whether to shuffle the dataset indices.
            drop_last (bool)     : whether to drop the last incomplete batch.
        """
        
        # set up instance variables
        self.data_source   = data_source                     # this is essentially the whole dataset
        self.num_samples   = len(data_source)                # length of the whole dataset
        self.indices       = list(range(self.num_samples))   # base indices. (shuffled) batches of these are returned
        self.schedule      = self._expand_schedule(schedule) # schedule dict but now with each batch size repeated * n
        self.current_epoch = 0                               # to keep track of the current epoch in training loop
        self.shuffle       = shuffle
        self.drop_last     = drop_last

    def _expand_schedule(self, schedule):
        """
        Expand the schedule to have a list of repeated batch size values. Returns a dict.
        e.g. {3 : 128, 2 : 256} -> {0 : 128, 1 : 128, 2 : 128, 3 : 256, 4 : 256}
        """
        
        expanded_schedule = {}
        cumulative_epochs = 0
        # dict.items() yields a tuple of (key, value)
        for nr_eps, bs in schedule.items():
            for i in range(cumulative_epochs, cumulative_epochs + nr_eps):
                # rebuilds the schedule dict. key i is the epoch nr i and value is the batch size for that epoch
                expanded_schedule[i] = bs
            cumulative_epochs += nr_eps
            
        return expanded_schedule

    # ... could also be replaced with something like batch_size_scheduler.step (so just step haha)
    def set_epoch(self, epoch):
        """
        update the current epoch when called in the training loop
        """
        
        self.current_epoch = epoch

    def _get_batch_size(self):
        """
        just convenience for getting the currend batch size from the expanded schedule depending on current epoch. If the epoch is larger than what is specified in the schedule, it just returns the last value.
        """
        
        # use dict.get(key) instead of dict[key] in order to be able to specify a default for when key is missing!
        return self.schedule.get(self.current_epoch, list(self.schedule.values())[-1])

    def __iter__(self):
        """
        this function is essentially responsible for returning a generator object. The yield statement is kind of like a return, but instead it just advances the loop further every time the next() method is called, or in a for loop. it kind of remebers the last time it was called.
        """
        
        # determine batch size
        batch_size = self._get_batch_size()
        
        # Shuffle before returning the generator object for the new epoch
        if self.shuffle:
            random.shuffle(self.indices) 

        # determines the start indices for each batch. range(start, stop, step). 
        # Note this is essentially only the index to find the indices in self.indices for one batch xD
        for start_idx in range(0, len(self.indices), batch_size):
            # grab all the indices (now those indices corresponding to samples) belonging to a back
            batch = self.indices[start_idx:start_idx + batch_size]
            # skip yielding the last semi-full batch if self.drop_last is true
            if len(batch) < batch_size and self.drop_last:
                continue
            # basically returns one element when it's called and then pauses the function execution until next()
            yield batch

    def __len__(self):
        """
        just returns the number of batches in the current epoch. changes by 1 depending on whether the last semi-full one is dropped or not.
        """
        
        batch_size = self._get_batch_size()
        if self.drop_last:
            return math.floor(self.num_samples / batch_size)
        else:
            return math.ceil(self.num_samples / batch_size)


# Example Dataset
class ExampleDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


# Example Usage
data = [i for i in range(51)]  # Example dataset
dataset = ExampleDataset(data)

# Schedule: {30 epochs at 128, 20 epochs at 256, 10 epochs at 512}
schedule = {2: 10, 3: 20, 4: 50}

# Instantiate the ScheduledBatchSampler
sampler = ScheduledBatchSampler(dataset, schedule, shuffle=True, drop_last=False)

# DataLoader
dataloader = DataLoader(dataset, batch_sampler=sampler)

# Simulate Training Loop
for epoch in range(15):  # simulate epochs
    print(f"Epoch {epoch + 1}")
    sampler.set_epoch(epoch)  # Update epoch for batch size scheduling
    for batch in dataloader:
        print(f"Batch with size: {len(batch)}")



Epoch 1
Batch with size: 10
Batch with size: 10
Batch with size: 10
Batch with size: 10
Batch with size: 10
Batch with size: 1
Epoch 2
Batch with size: 10
Batch with size: 10
Batch with size: 10
Batch with size: 10
Batch with size: 10
Batch with size: 1
Epoch 3
Batch with size: 20
Batch with size: 20
Batch with size: 11
Epoch 4
Batch with size: 20
Batch with size: 20
Batch with size: 11
Epoch 5
Batch with size: 20
Batch with size: 20
Batch with size: 11
Epoch 6
Batch with size: 50
Batch with size: 1
Epoch 7
Batch with size: 50
Batch with size: 1
Epoch 8
Batch with size: 50
Batch with size: 1
Epoch 9
Batch with size: 50
Batch with size: 1
Epoch 10
Batch with size: 50
Batch with size: 1
Epoch 11
Batch with size: 50
Batch with size: 1
Epoch 12
Batch with size: 50
Batch with size: 1
Epoch 13
Batch with size: 50
Batch with size: 1
Epoch 14
Batch with size: 50
Batch with size: 1
Epoch 15
Batch with size: 50
Batch with size: 1


In [26]:
def iterator():
    for el in [1, 2, 3, 4]:
        yield el

a = iterator()
print(next(a))
print(next(a))


1
2
